# atari_reproducibility — DQN reproducibility on two Atari games (10M frames)

Four plots, one per `(setting, game)` pair: the `dqn_atari` and
`dqn_real_atari` settings each sweep `ENV_HYPERS.GAME` over `battle_zone`,
`ms_pacman`, and each run 3 times with the same config and PRNG seed
(`Component.seeds=[0]`). Every replicate is its own component
(`<setting>_replicate_<i>`), so identical runs are stored separately
instead of deduping into one. 12 runs total, 4 plots. Each plot draws its 3 replicates as
separate colored curves, clipped to our 10M-frame training window - no mean,
no CI band, no Reference DQN overlay. This is the reproducibility check:
replicate curves that track each other closely mean re-running the same
seed gives the same result. (Atari is cluster-scale - this notebook expects
results produced elsewhere; see the experiment README.)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_HERE = Path.cwd()
_EXP_DIR = _HERE if (_HERE / "config.py").exists() else Path("experiments/atari_reproducibility")
sys.path.insert(0, str(_EXP_DIR))
sys.path.insert(0, str(_EXP_DIR.resolve().parents[1]))

from experiment import load_result, load_runs
from analysis.plotting import seed_grids_for, style

from config import EXPERIMENT

FONT_SIZE = 20   # 2x matplotlib's default (10), passed to every text call below

COMPONENT_LABELS = {"dqn_atari": "Standard", "dqn_real_atari": "Real Atari"}


def grid_for(component, n=500):
    """A shared [0, total_steps] timestep grid from any run's length."""
    df = load_runs(EXPERIMENT, component)
    total_steps = len(load_result(EXPERIMENT, component, df["run_id"][0])["reward"])
    return np.linspace(0, total_steps, n)


def replicate_components(setting):
    """The component names holding one setting's replicates, in order."""
    prefix = f"{setting}_replicate_"
    return [c.name for c in EXPERIMENT.components if c.name.startswith(prefix)]


def plot_title(setting, game):
    """The plot title for one (setting, game) pair: e.g. "Battle Zone (Standard)"."""
    return f"{game.replace('_', ' ').title()} ({COMPONENT_LABELS[setting]})"


def plot_replicates(ax, grid, stack):
    """Draw each replicate as its own curve, in its own color."""
    for i, row in enumerate(stack):
        ax.plot(grid, row, lw=1.5, label=f"Replicate {i}")


In [ ]:
HERE = Path.cwd()
RESULTS = HERE / "results" if (HERE / "results").exists() else _EXP_DIR / "results"

SETTINGS = ["dqn_atari", "dqn_real_atari"]

GRIDS = {s: grid_for(replicate_components(s)[0]) for s in SETTINGS}

# GRID is in env steps; scale to frames (steps x frameskip) for the x-axis.
FRAMESKIP = load_runs(EXPERIMENT, replicate_components(SETTINGS[0])[0])[
    "ENV_HYPERS.FRAMESKIP"
][0]
FRAME_GRIDS = {s: GRIDS[s] * FRAMESKIP for s in SETTINGS}
FRAME_TICKS = np.array([2, 4, 6, 8, 10]) * 1_000_000
FRAME_TICK_LABELS = [f"{t // 1_000_000}M" for t in FRAME_TICKS]


In [ ]:
def game_stack(setting, game):
    """One row per replicate: its return curve on `game`."""
    rows = []
    for component in replicate_components(setting):
        df = load_runs(EXPERIMENT, component)
        run_ids = df.filter(df["ENV_HYPERS.GAME"] == game)["run_id"].to_list()
        rows.append(seed_grids_for(EXPERIMENT, component, GRIDS[setting], run_ids=run_ids))
    return np.vstack(rows)


figures = {}
for setting in SETTINGS:
    games = load_runs(EXPERIMENT, replicate_components(setting)[0])["ENV_HYPERS.GAME"]
    for game in games.unique(maintain_order=True):
        stack = game_stack(setting, game)

        fig, ax = plt.subplots(figsize=(9, 6))   # 2:3 height:width
        plot_replicates(ax, FRAME_GRIDS[setting], stack)
        ax.set_title(plot_title(setting, game), fontsize=FONT_SIZE)
        ax.legend(loc="upper left", frameon=False, fontsize=FONT_SIZE)
        style(ax, xlabel="Frames")   # score range differs per game, so let it auto-scale
        ax.set_ylabel(
            "Return", rotation=90, ha="center", va="center",
            labelpad=20, fontsize=FONT_SIZE,
        )
        ax.set_xlabel(ax.get_xlabel(), fontsize=FONT_SIZE)
        ax.set_xticks(FRAME_TICKS)
        ax.set_xticklabels(FRAME_TICK_LABELS)
        ax.tick_params(labelsize=FONT_SIZE)
        fig.tight_layout()
        figures[f"{setting}_{game}"] = fig

plt.show()


In [ ]:
PLOTS_DIR = RESULTS.parent / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")
